## What is LLM Gateway?
LLM Gateway is smart middleware layer between your application and multiple llm providers (OpenAI, Gemini, Anthropic, Grog etc.,)

### Benefits with Gateway
* One Unified Gateway
* Automatic fallback
* Swap models with configuration change
* Centralized logging, rate limiting, cost tracking
* Cache repeated queries


# Installation and Setup
We are going use
* LiteLLM - popular open source LLM gateway
* LangChain - to develop agentic workflows on top of llm gateway

In [2]:
! pip install -q litellm langchain langchain-community langchain-openai python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

from litellm import completion

In [4]:
import litellm
litellm.suppress_debug_info = True

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

print("OpenAI key loaded: ", "Available" if os.getenv("OPENAI_API_KEY") else "Not Available")
print("Groq key loaded: ", "Available" if os.getenv("GROQ_API_KEY") else "Not Available")
print("Gemini key loaded: ", "Available" if os.getenv("GEMINI_API_KEY") else "Not Available")

OpenAI key loaded:  Available
Groq key loaded:  Available
Gemini key loaded:  Available


# LLM Example - Unified API
The pain point is every provider has a different SDK

LiteLLM provides **completion()** method works with all of them. 

In [16]:
from litellm import completion

response_openai = completion(
    model = "gpt-4o-mini",
    messages = [{"role": "user", "content": "What is the capital of France?"}]
)
print("OpenAI response:", response_openai.choices[0].message.content)

response_groq = completion(
    model = "groq/llama-3.3-70b-versatile",
    messages = [{"role": "user", "content": "What is the capital of France?"}]
)
print("Groq response:", response_groq.choices[0].message.content)

OpenAI response: The capital of France is Paris.
Groq response: The capital of France is Paris.


**Notice**: Same code, different providers. This alone is huge - we can switch providers with a string change.


In [17]:
from litellm import completion

prompt = "What is the capital of France?"
providers = [("OpenAI", "gpt-4o-mini"), ("Groq", "groq/llama-3.3-70b-versatile"), ("Gemini", "gemini-1.5-pro"),("Anthropic", "claude-claude-3-5-haiku-20241022")]

for label, model in providers:
    try:
        res = completion(
            model=model,
            messages=[{"role": "user", "content": prompt}]
        )
        print(f"{label}: {res.choices[0].message.content}")
    except Exception as e:
        print(f"{label} error: {e}")

OpenAI: The capital of France is Paris.
Groq: The capital of France is Paris.
Gemini error: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=gemini-1.5-pro
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
Anthropic error: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=claude-claude-3-5-haiku-20241022
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers


In [18]:
from litellm import completion

res = completion(
    model = "google/gemini-1.5-flash",
    messages = [{"role": "user", "content": "What is the capital of France?"}],
    fallbacks = [
        "openai/gpt-4o-mini",
        "groq/llama-3.3-70b-versatile",]
)
print("response:", res.choices[0].message.content)
print("provider used:", res.model)

16:36:34 - LiteLLM:ERROR: fallback_utils.py:68 - Fallback attempt failed for model google/gemini-1.5-flash: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=google/gemini-1.5-flash
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
Traceback (most recent call last):
  File "c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\litellm\litellm_core_utils\fallback_utils.py", line 58, in async_completion_with_fallbacks
    response = await litellm.acompletion(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\litellm\utils.py", line 2099, in wrapper_async
    raise e
  File "c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\litellm\utils.py", line 1898, in wrapper_async
    result = await original_fu

Task was destroyed but it is pending!
task: <Task pending name='Task-158' coro=<LoggingWorker._worker_loop() running at c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\litellm\litellm_core_utils\logging_worker.py:110>>


response: The capital of France is Paris.
provider used: gpt-4o-mini-2024-07-18


## Cost Tracking
LiteLLM **calculates the cost of tokens** of every call using its built-in pricing database.

In [20]:
from litellm import completion, completion_cost

response = completion(
    model = "openai/gpt-4o-mini",
    messages = [{"role": "user", "content": "What is the capital of France?"}])

cost = completion_cost(completion_response=response)

print("Response:", response.choices[0].message.content)
print("\n Input tokens:", response.usage.prompt_tokens)
print(" Output tokens:", response.usage.completion_tokens)

print(f" Total cost (USD): ${cost:.8f}")

Response: The capital of France is Paris.

 Input tokens: 14
 Output tokens: 7
 Total cost (USD): $0.00000630


## Caching: If users ask the same question, we dont need to call LLM those many times.
Enable in-memory caching with one line.

In [21]:
import litellm

litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm.async_success_callback = []
litellm.async_failure_callback = []

litellm.cache = None

print("Litellm state reset - ready for clean caching demo.")


Litellm state reset - ready for clean caching demo.


In [22]:
import litellm
import time
from litellm import completion
from litellm.caching import Cache

litellm.cache = Cache(type="local")

prompt = "What does LLM stand for? "

start = time.time()
res1 = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t1 = time.time() - start
print(f"First call (API):   {t1:.2f}s — {res1.choices[0].message.content}")

start = time.time()
r2 = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t2 = time.time() - start
print(f"Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\n Speedup: {t1/t2:.1f}x faster, and ZERO cost on the second call!")

First call (API):   3.54s — LLM can stand for a few different things depending on the context. The most common meanings are:

1. **Large Language Model**: In the field of artificial intelligence and natural language processing, LLM refers to advanced models that have been trained on vast amounts of text data to understand and generate human-like language. Examples include OpenAI's GPT series.

2. **Master of Laws (Legum Magister)**: In academia, LLM is a postgraduate academic degree in law.

If you have a specific context in mind, please let me know!
Second call (cache): 0.0157s — LLM can stand for a few different things depending on the context. The most common meanings are:

1. **Large Language Model**: In the field of artificial intelligence and natural language processing, LLM refers to advanced models that have been trained on vast amounts of text data to understand and generate human-like language. Examples include OpenAI's GPT series.

2. **Master of Laws (Legum Magister)**: In 

## Smart Routing - Assign the model based on activity

* Coding tasks - Claude sonnet
* Cheap summaries - GPT-40-mini
* Fast replies - Groq llama
* Complex reasoning - Claude Opus

Use Litellm's Router to define routing rules.

In [24]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",                              
        "litellm_params": {
            "model": "gpt-4o",                                      
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    }
]

router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "What is the capital of France?"}]
)

code_response = router.completion(
    model="smart-coding",  
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print(" Fast/cheap (Groq): ", fast_response.choices[0].message.content[:150])
print("\n Smart/coding (GPT-4o):\n", code_response.choices[0].message.content[:300])


 Fast/cheap (Groq):  The capital of France is Paris.

 Smart/coding (GPT-4o):
 Certainly! You can reverse a string in Python by defining a function that utilizes slicing. Here is a simple function to accomplish this:

```python
def reverse_string(s):
    """Return the reverse of the input string s."""
    return s[::-1]

# Example usage
original_string = "Hello, World!"
revers


In [25]:
from litellm import Router
import os

model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "gpt-4o",
            "api_key": os.getenv("OPENAI_API_KEY"),
        },
        "model_info": {"id": "openai-gpt4o"}
    },
    
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq-llama-70b"}
    },
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        openai-gpt4o            1778 ms   Hello! How can I assist you today?
#2        openai-gpt4o            1026 ms   Hello! How can I assist you today? 
#3        groq-llama-70b           196 ms   Hello. You've requested 3, could yo
#4        openai-gpt4o             728 ms   Hello! How can I assist you today? 
#5        openai-gpt4o             576 ms   Hello! How can I assist you today?
#6        groq-llama-70b           249 ms   Hello. You've requested 6, but I'm 


## Different types of Strategies
* least-busy
* latency-based-routing
* cost-based-routing

## Observability - log every single call
We must log every single call - prompt, response, latency, cost, user-id etc.,

In [26]:
import litellm
from litellm import completion

call_logs = []

def log_success(kwargs, completion_response, start_time, end_time):
    call_logs.append({
        "model": kwargs.get("model"),
        "prompt": kwargs["messages"][-1]["content"][:60],
        "input_tokens": completion_response.usage.prompt_tokens,
        "output_tokens": completion_response.usage.completion_tokens,
        "latency_sec": round((end_time - start_time).total_seconds(), 2),
        "cost_usd": kwargs.get("response_cost", 0),
        "user": kwargs.get("user", "anonymous")
    })

def log_failure(kwargs, completion_response, start_time, end_time):
    print(" Call failed:", kwargs.get("exception"))

litellm.success_callback = [log_success]
litellm.failure_callback = [log_failure]

for q, user in [
    ("What is RAG?", "satya"),
    ("Explain transformers.", "emploee"),
    ("What is fine-tuning?", "satya"),
]:
    completion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": q}],
        user=user
    )

# Review the audit log
import json
print(json.dumps(call_logs, indent=2, default=str))

[]


## Integating with LangChain
Langchain for orchestration and LiteLLM as unified LLM backend.


In [27]:
! pip install -q langchain-litellm


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatLiteLLM(model="gpt-4o-mini", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor. Be concise."),
    ("user", "{question}")
])

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"question": "What is an LLM Gateway in 3 bullets?"})
print(answer)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x00000242A622B620>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x00000242A6D5B650>, 1116292.906)])']
connector: <aiohttp.connector.TCPConnector object at 0x00000242A622BA70>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x00000242A622E240>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x00000242A6D5BEF0>, 1116477.875)])']
connector: <aiohttp.connector.TCPConnector object at 0x00000242A4B4AB70>


- **Definition**: An LLM Gateway is an interface or platform that facilitates access to Large Language Models (LLMs) for various applications, enabling users to leverage AI capabilities without deep technical expertise.

- **Functionality**: It allows users to input queries or prompts and receive generated responses from LLMs, often providing features like customization, API integration, and user-friendly dashboards.

- **Use Cases**: Commonly used in customer support, content generation, language translation, and other natural language processing tasks, making advanced AI accessible to businesses and developers.


### We can swap the exising model with other models

## Example - LangChain chain with fallbacks

In [29]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

primary = ChatLiteLLM(model="gpt-4o1")

fallback1 = ChatLiteLLM(model="gpt-4o-mini", temperature=0.2)
fallback2 = ChatLiteLLM(model="groq/llama-3.3-70b-versatile", temperature=0.2)

# LangChain's .with_fallbacks() chains them together
llm = primary.with_fallbacks([fallback1, fallback2])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI engineer. Always reply in JSON: {{\"answer\": ...}}"),
    ("user", "{question}")
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"question": "What are the top 3 benefits of an LLM Gateway?"})
print(result)

 Call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=gpt-4o1
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
{
  "answer": {
    "benefits": [
      {
        "title": "Centralized Access",
        "description": "An LLM Gateway provides a single point of access for multiple language models, simplifying integration and management for developers."
      },
      {
        "title": "Scalability",
        "description": "It allows organizations to scale their usage of language models efficiently, enabling them to handle varying workloads without significant infrastructure changes."
      },
      {
        "title": "Enhanced Security",
        "description": "An LLM Gateway can implement security protocols and access controls, ensuring that sensitive data is protected while interacting with langua